In [ ]:
import pandas as pd
import pickle
import torch
import os
import torch.nn.functional as F
import config

from tqdm.auto import tqdm

from torch.utils.data import  DataLoader
from src.metric import *
from sentence_transformers import SentenceTransformer,util,CrossEncoder


from src.bi_encoder_training import get_resume_embedding,get_jd_embedding,cosent_loss,compute_batch_embeddings
from src.cross_encoder_training import compute_batch_scores
from src.datasets import ResumeJDDataset



In [ ]:
path=config.CLEANED_DATA_DIR

In [ ]:
with open(os.path.join(path,'train_df.pkl'),'rb') as f:
    train_df=pickle.load(f)
    
with open(os.path.join(path,'val_df.pkl'),'rb') as f:
    val_df=pickle.load(f)
        
with open(os.path.join(path,'test_df.pkl'),'rb') as f:
    test_df=pickle.load(f)
    

In [ ]:
BATCH_SIZE=config.CHUNK_BATCH_SIZE

In [ ]:
bi_encoder = SentenceTransformer(os.path.join(config.CHUNKED_MODEL_DIR,'bi_encoder_chunks'))

In [ ]:
unique_resume=train_df['resume_text'].drop_duplicates().tolist()

resume_embs=[]

for resume in tqdm(unique_resume):
    resume_embs.append(get_resume_embedding(bi_encoder,resume,None))

pos_rows=train_df[train_df['label']==2]

In [ ]:
hard_neg_rows=[]
top_k_hard_neg=20

for _,row in pos_rows.iterrows():
    jd=row['job_description_text']
    pos_resume=row['resume_text']

    jd_emb=get_jd_embedding(bi_encoder,jd)

    hits=util.semantic_search(jd_emb,resume_embs,top_k=top_k_hard_neg)
    hits=hits[0]

    added=0
    for hit in hits:
        idx=hit['corpus_id']
        candidate_resume=unique_resume[idx]
        if candidate_resume==pos_resume:
            continue
            
        existing=train_df[(train_df["job_description_text"] == jd)
            &
            (train_df["resume_text"] == candidate_resume)]
        
        if len(existing)>0 and existing.iloc[0]['label']>0:
            continue

        hard_neg_rows.append({
            "job_description_text": jd,
            "resume_text": candidate_resume,
            "label": 0
        })

        added+=1
        if added>3:
            break

hard_neg_df=pd.DataFrame(hard_neg_rows)
print("Hard Negative Rows:", len(hard_neg_df))

In [ ]:
enhanced_train=pd.concat([train_df,hard_neg_df])
print(len(enhanced_train))

In [ ]:
#Retrain bi-encoder with enhanced data 

In [ ]:
labels=[float(config.label_to_score[label]) for label in enhanced_train['label']]
train_dataset=ResumeJDDataset(enhanced_train['resume_text'].values,enhanced_train['job_description_text'].values,labels)

bi_dataloader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)

In [ ]:
min_delta=0.01
count=0
best_score=float('-inf')
epochs=3
patience=2

best_bi_encoder_path=os.path.join(config.HD_MODEL_DIR,'bi_encoder_hard_negatives')
os.makedirs(config.HD_MODEL_DIR,exist_ok=True)

In [ ]:
optimizer = torch.optim.AdamW(bi_encoder.parameters(), lr=2e-5)

In [ ]:
for epoch in range(epochs):
    
    print(f"Epoch {epoch+1}/{epochs}")
    bi_encoder.train()
    
    total_loss=0
    
    progress_bar = tqdm(bi_dataloader, desc="Training")
    
    for resumes,jds,labels in progress_bar:
        optimizer.zero_grad()
        
        resume_embs, jd_embs=compute_batch_embeddings(bi_encoder,resumes,jds)
        
        labels=labels.to(config.device)
        loss=cosent_loss(resume_embs,jd_embs,labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")
        
    print("Final Loss:",total_loss/len(progress_bar))
        
        
    #Validation    
    bi_encoder.eval()
    
    with torch.no_grad():
            
        val_resume_embs, val_jd_embs=compute_batch_embeddings(bi_encoder,val_df['resume_text'].values,val_df['job_description_text'].values)
        
        scores=F.cosine_similarity(val_resume_embs,val_jd_embs,dim=1)
        scores=scores.cpu().numpy()
        
        metrics=model_evaluation(scores,val_df,"job_description_text")
        print("NDCG:", metrics["ndcg_val"])
        print("MAP:", metrics["map_score"])
        
        final_score=0.6*metrics["ndcg_val"]+0.3*metrics["map_score"]+0.1*metrics["mrr_score"]
        
    if final_score>best_score+min_delta:
        best_score=final_score
        bi_encoder.save(best_bi_encoder_path)
        count=0
    else:
        count+=1

    if count==patience:
        print("Early stopping triggered.")
        break
                
            
    

In [ ]:
bi_encoder=SentenceTransformer(best_bi_encoder_path,device=config.device)

In [ ]:
bi_encoder.eval()

with torch.no_grad():
        
    val_resume_embs,val_jd_embs=compute_batch_embeddings(bi_encoder,test_df['resume_text'].values,test_df['job_description_text'].values)
    
    scores=F.cosine_similarity(val_resume_embs,val_jd_embs,dim=1)
    scores=scores.cpu().numpy()
    
    metrics=model_evaluation(scores,test_df,"job_description_text")

In [ ]:
print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])

Cross Encoder

In [ ]:
cross_encoder = CrossEncoder(os.path.join(config.CHUNKED_MODEL_DIR, 'cross_encoder_chunks'),num_labels=1,device=config.device)

In [ ]:
labels=[float(config.label_to_score[label]) for label in enhanced_train['label']]
train_dataset=ResumeJDDataset(enhanced_train['resume_text'].values,enhanced_train['job_description_text'].values,labels)

train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)

In [ ]:
cross_enoder_best_model_path=os.path.join(config.HD_MODEL_DIR,'cross_encoder_hard_negatives')
os.makedirs(config.HD_MODEL_DIR,exist_ok=True)

In [ ]:
optimizer = torch.optim.AdamW(cross_encoder.parameters(), lr=2e-5)
loss_fn = torch.nn.MSELoss()

In [ ]:
min_delta=0.01
count=0
best_score=float('-inf')
epochs=3
patience=2

In [ ]:
for epoch in range(epochs):
    
    print(f"Epoch {epoch+1}/{epochs}")
    cross_encoder.train()
    
    total_loss=0
    
    progress_bar = tqdm(train_loader, desc="Training")
    
    for resumes,jds,labels in progress_bar:
        optimizer.zero_grad()
        
        scores=compute_batch_scores(cross_encoder,resumes,jds)
        labels=labels.to(config.device)
        
        loss=loss_fn(scores,labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss+=loss.item()
        
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")
        
    average_loss=total_loss/len(progress_bar)
    print("Average Loss:",average_loss)
        
        
    #Validation    
    cross_encoder.eval()
    
    with torch.no_grad():
            
        val_scores=compute_batch_scores(cross_encoder,val_df['resume_text'].values,val_df['job_description_text'].values)
        val_scores=val_scores.cpu().numpy()
       
        metrics=model_evaluation(val_scores,val_df,"job_description_text")
        print("NDCG:", metrics["ndcg_val"])
        print("MAP:", metrics["map_score"])
        
        final_score=0.6*metrics["ndcg_val"]+0.3*metrics["map_score"]+0.1*metrics["mrr_score"]
        
    if final_score>best_score+min_delta:
        best_score=final_score
        cross_encoder.save(cross_enoder_best_model_path)
        count=0
    else:
        count+=1

    if count==patience:
        print("Early stopping triggered.")
        break
                
            
    

In [ ]:
cross_encoder=CrossEncoder(cross_enoder_best_model_path,num_labels=1,device=config.device)

In [ ]:
cross_encoder.eval()
with torch.no_grad():
    scores=compute_batch_scores(cross_encoder,test_df['resume_text'].values,test_df['job_description_text'].values)
    scores=scores.cpu().numpy()
    metrics=model_evaluation(scores,test_df,"job_description_text")

In [ ]:
print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])